In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print(os.listdir("/content/drive/MyDrive"))
print(os.chdir("/content/drive/MyDrive/Machine Learning Club (datasets)"))
print(os.listdir(os.getcwd()))

In [ ]:
# Data Collection
df = pd.read_csv("netflix_titles.csv")

In [ ]:
# Data cleaning and preprocessing
print(df.head())
print(df.tail())
print(df.shape)
print(df.columns)
print(f"info: {df.info()}")

In [ ]:
print(df.isnull().sum())
print(df.isnull().mean() * 100)

In [ ]:
dataset_null_values = pd.DataFrame({
    "null_values": df.isnull().sum(),
    "null_percentage": df.isnull().mean() * 100
})
print(dataset_null_values.sort_values("null_values", ascending=False))

In [ ]:
df_copy = df.copy()
df.head()

Now I will just inspect each column and see how could I clean it. Just messing around with the columns.


In [ ]:
df["type"].unique()
# Just two types, we can use one-hot encoding on that column

In [ ]:
print(df["director"].unique())
print(df["director"].unique().shape)

In [ ]:
print(df["cast"].head(10))
print(df["cast"][1].split(", "), len(df["cast"][1].split(", ")))
# Let's see if we can use one-hot encoding on them

In [ ]:
unique_casts = []
count = 0
for i in df["cast"]:
    if not pd.isnull(i):
        for cast in i.split(", "):
            if cast not in unique_casts:
                unique_casts.append(cast)    
    count += 1
print(unique_casts, len(unique_casts))
# Too Large unique actors or casts

In [ ]:
print(df["country"].unique(), df["country"].unique().shape)
# Too much countries.

In [ ]:
print(df["date_added"].head(10))
# Same format, we can simply transform it to a datetime format

In [ ]:
print(df["rating"].head(10))
print(df["rating"].unique(), df["rating"].unique().shape)
# We can use one-hot encoding here but I don't think that this one-hot encoding will represent anything or be useful so I will let it as it is.

In [ ]:
print(df["duration"].head(10))
print(df["duration"].unique())
# Here the duration isn't really a simple duration but a mix of whether it's seasons (TV Show) or the real duration (Movie). I can use one hot encoding here to represent either the duration of a movie or the number of seasons of a series

In [ ]:
unique_types = []
count = 0
for i in df["listed_in"]:
    if not pd.isnull(i):
        for type in i.split(", "):
            if type not in unique_types:
                unique_types.append(type)    
    count += 1
print(unique_types, len(unique_types))
# This has 42 unique types so we can use one-hot encoding. 42 columns seem kind of moderate and will be significantly more useful than this listed_in column so I think that a one-hot encoding is a better approach here.

Here's the plan for each column:

1. show_id: doesn't have any importance. Action: Remove.
2. type: Do have importance and is classified into 2 types. Action: One-hot encoding.
3. title: Do have a little bit of importance and seems clean. Action: Nothing.
4. director: This column is important but we cannot fill the none values with mean or median due to it being a string. Action: Fill nan with mode.
5. cast: There is two options: Either we use one-hot encoding for each member in the cast and make it binary (either this one person is in the cast or not), but that will blow the columns and will make it contain too many columns, or to simply transform the cast to make it only indicates the number of casts that are in the movie/series. Action: Get the number of actors for each cast.
6. country: The same thing as director column.
7. date_added: Transform datatype into date format and then use mode to fill the none values.
8. release_year: As it is.
9. rating: The same thing as director & country columns.
10. duration: Use one hot encoding to split the duration into 2 types: either seasons or duration.
11. listed_in: The same thing as cast.
12. description: As it is.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded_array = one_hot_encoder.fit_transform(df_copy[["type"]])
encoded_df = pd.DataFrame(
    encoded_array,
    columns=one_hot_encoder.get_feature_names_out(["type"]),
    index=df_copy.index
)
encoded_df
df_copy["movie"] = encoded_df["type_Movie"]
df_copy["tv show"] = encoded_df["type_TV Show"]

In [ ]:
# duration column
count = 0
df_copy["Seasons"] = 0
for row in df_copy["duration"]:
    if not pd.isnull(row):
        if "Seasons" in row or "Season" in row:
            df_copy["Seasons"][count] = row.split(" ")[0]
            df_copy["duration"][count] = 0
        else:
            df_copy["duration"][count] = row.split(" ")[0]
    count += 1

In [ ]:
# date-added column
df_copy["date_added"] = pd.to_datetime(
    df_copy["date_added"].str.strip(),
    errors="coerce",
)

In [ ]:
# Cast Column
count = 0
for row in df_copy["cast"]:
    if not pd.isnull(row): 
        df_copy["cast"][count] = len(df_copy["cast"][count].split(", "))
    count += 1
print(count)

In [ ]:
# Listed_in
count = 0
unique_types = []
for row in df_copy["listed_in"]:
    if not pd.isnull(row): 
        for i in df_copy["listed_in"][count].split(", "):
            if i not in unique_types:
                unique_types.append(i)
    count += 1

In [ ]:
for i in unique_types:
    df_copy[i] = 0
count = 0
for row in df_copy["listed_in"]:
    if not pd.isnull(row):
        for i in df_copy["listed_in"][count].split(", "):
            if i in unique_types:
                df_copy[i][count] = 1
    count += 1

In [ ]:
df_copy["rating"] = df_copy["rating"].fillna(df_copy["rating"].mode())

In [ ]:
df_copy = df_copy.drop(columns=["title", "show_id", "listed_in", "type"])
df_copy["rating"] = df_copy["rating"].fillna(df_copy["rating"].mode()[0])
df_copy["director"] = df_copy["director"].fillna(df_copy["director"].mode()[0])
df_copy["duration"] = df_copy["duration"].fillna(df_copy["duration"].median())
df_copy["Seasons"] = df_copy["Seasons"].fillna(df_copy["Seasons"].mode()[0])
df_copy["date_added"] = df_copy["date_added"].fillna(df_copy["date_added"].median())
df_copy["country"] = df_copy["country"].fillna(df_copy["country"].mode()[0])
df_copy["cast"] = df_copy["cast"].fillna(df_copy["cast"].median())

In [ ]:
df_copy.duplicated().sum()
df_copy = df_copy.drop_duplicates()

In [ ]:
df_copy.head()

In [ ]:
dataset_null_values = pd.DataFrame({
    "null_values": df_copy.isnull().sum(),
    "null_percentage": df_copy.isnull().mean() * 100
})
print(dataset_null_values.sort_values("null_values", ascending=False))

In [ ]:
df_copy.to_csv("movie_reviews_cleaned.csv", index=False)
files.download("movie_reviews_cleaned.csv")

In [ ]:
df.iloc[0]
df.loc[0]

In [ ]:
df[df["type"] == "Movie"].head(10)

In [ ]:
df.sort_values(by="duration", ascending=False).head(10)

In [ ]:
df.groupby("release_year")["duration"].mean()

In [ ]:
df.dtypes

In [ ]:
pd.to_datetime(df["release_year"])

In [ ]:
df.isnull().head()

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().mean() * 100

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})
missing_summary.sort_values(by="Missing Percentage", ascending=False)

In [ ]:
clean_df = df.copy()

clean_df["duration"] = clean_df["duration"].fillna(clean_df["duration"].median())

In [ ]:
clean_df.duplicated().sum()
clean_df[clean_df.duplicated(keep=False)].head(5)

In [ ]:
clean_df.drop_duplicates()

In [ ]:
print(f"release_year uniqueness: {clean_df["release_year"].unique()}")